PROMPT<br>

Help me write an SQL query that segments customers based on their purchasig behavior for a new feature rollout.The database has three tables: user_activity (user_id, last_login_date, feature_usage_count, account_type), transactions (transaction_id, user_id, transaction_date, amount, platform), and user_preferences (user_id, communication_preference, interface_theme, notification_settings). The query should identify active users (logged in within the last 30 days), filter high-value customers (top 20% by total spending), and return user preference trends for those selected customers

In [ ]:
# Code

WITH active_users AS (
    SELECT user_id
    FROM user_activity
    WHERE last_login_date >= CURRENT_DATE - INTERVAL '30 days'
),

spending AS (
    SELECT user_id,
           SUM(amount) AS total_spent,
           NTILE(5) OVER (ORDER BY SUM(amount) DESC) AS spending_group
    FROM transactions
    GROUP BY user_id
),

high_value_users AS (
    SELECT user_id
    FROM spending
    WHERE spending_group = 1
),

segmented_users AS (
    SELECT a.user_id
    FROM active_users a
    INNER JOIN high_value_users h
        ON a.user_id = h.user_id
)

SELECT
    s.user_id,
    p.communication_preference,
    p.interface_theme,
    p.notification_settings
FROM segmented_users s
LEFT JOIN user_preferences p
    ON s.user_id = p.user_id;

Follow up prompts<br>

1.Add customer segmentation labels such as Active, High Value<br>
2.query to improve performance using optimized joins and indexing assumptions<br>
3.Each user monthly spending trends  for deeper behavioral analysis<br>

In [ ]:
# final code
WITH active_users AS (
    SELECT
        user_id,
        last_login_date
    FROM user_activity
    WHERE last_login_date >= DATE_SUB(CURDATE(), INTERVAL 30 DAY)
),

spending AS (
    SELECT
        user_id,
        SUM(amount) AS total_spent
    FROM transactions
    GROUP BY user_id
),

high_value_customers AS (
    SELECT
        user_id
    FROM (
        SELECT
            user_id,
            total_spent,
            NTILE(5) OVER (ORDER BY total_spent DESC) AS spending_rank
        FROM spending
    ) ranked
    WHERE spending_rank = 1
),

segmented_users AS (
    SELECT
        a.user_id,
        CASE
            WHEN h.user_id IS NOT NULL THEN 'Active + High Value'
            ELSE 'Active Only'
        END AS segment
    FROM active_users a
    LEFT JOIN high_value_customers h
        ON a.user_id = h.user_id
)

SELECT
    s.user_id,
    s.segment,
    p.communication_preference,
    p.interface_theme,
    p.notification_settings
FROM segmented_users s
LEFT JOIN user_preferences p
    ON s.user_id = p.user_id;

1.How were percentile calculations handled?

window functions, especially NTILE(5),It divides users into equal groups based on spending,grouping the top 20% (highest spenders) by selecting the highest-ranked group (e.g., rank = 1 in NTILE(5)).

2.What approaches to date filtering were used?

relative date conditions to capture recent activity. Common methods include:last_login_date >= CURRENT_DATE - INTERVAL '30 days' (PostgreSQL style),DATE_SUB(CURDATE(), INTERVAL 30 DAY) (MySQL style).It made sure that only active users within the last 30 days are captured.

3.How was the query optimized?

CTE broke the code into steps;finding the active users and how much they spend,then figured out who the best customers were using percentiles,then combined the steps.

